# KPI 3 — Revenue by Product / Category (Bestsellers + Slow-Movers)

**requirements.md (Sales & Revenue, KPI 3):** "Revenue by Product/Category: Product performance; identify slow-movers and bestsellers."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_3_revenue_by_product_category` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cell below once (or after any change to the fact/dim definitions).

**Definitions (state formulas):**

* **Grain:** one row per **product** (`product_id`) — all 160 catalog products have ≥ 1 order line with `order_date_valid = true` (verified; 0 null `product_sk` in the fact), so the view covers the full catalog
* **Gross revenue** = `SUM(line_total)` over ALL statuses with `order_date_valid = true`; **net revenue** = same minus `status = 'cancelled'` (KPI 1 bases, so totals reconcile exactly with KPI 1/KPI 2)
* **Units sold** = `SUM(quantity)`; 472 valid-date lines have null `quantity` (known silver issue — never null together with `unit_price`), they are excluded from the sum but still counted in `line_count` and revenue
* Aggregation is by `product_id` (business key) via the SCD2 dim join on `product_sk`, so a future re-versioned product cannot split into two rows; `sku` / `name` / `category` / `is_active` come from the **current** dim version (`__END_AT IS NULL`). Category revenue = `GROUP BY category` over this view
* **Bestsellers / slow-movers:** ranked by net revenue DESC (`ROW_NUMBER`, tie-break `product_id`); `is_bestseller` = rank ≤ `CEIL(0.2 × product_count)` (top 20%, same convention as KPI 2's Pareto); `is_slow_mover` = rank > `FLOOR(0.8 × product_count)` (bottom 20%)

**Verified 2026-11-19:** 160 rows = 160 distinct products (full catalog, all sold on valid dates); net total $28,333,498.48 / gross $30,723,269.53 — **exact** KPI 1 / KPI 2 reconciliation; 32 bestsellers (ranks 1–32) and 32 slow-movers (ranks 129–160), 0 overlap, ranks 1..160 contiguous; all 160 share %s + ranks + flags match an independent self-join recompute; category rollup matches preflight exactly (furniture 40.54% → cleaning 1.52%). Headlines: #1 bestseller KIT-0123 (kitchen, $4.74M net, 30,057 units); slowest OFF-0040 (office supplies, $1,936 net). Tier concentration: 17 of 32 bestsellers are furniture; 29 of 32 slow-movers are office supplies / packaging / cleaning.

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_3_revenue_by_product_category
TBLPROPERTIES (
  'comment' = 'KPI 3 - Revenue by Product/Category (requirements.md: product performance; identify slow-movers and bestsellers). Grain: one row per product_id (all 160 catalog products sold on valid dates; 0 null product_sk in the fact). Gross = SUM(line_total) ALL statuses, order_date_valid = true; net = same excluding status = ''cancelled'' (KPI 1 bases). Aggregated by product_id (business key) via the SCD2 dim join on product_sk so re-versioning cannot split a product; sku/name/category/is_active from the current dim version (__END_AT IS NULL). Units sold = SUM(quantity) - 472 valid-date lines with null quantity excluded from units but kept in revenue/line_count. Bestsellers/slow-movers: ROW_NUMBER by net revenue DESC (tie-break product_id); is_bestseller = rank <= CEIL(0.2 * product_count); is_slow_mover = rank > FLOOR(0.8 * product_count). Category revenue = GROUP BY category over this view.'
)
AS
WITH per_sk AS (
  SELECT
    f.product_sk,
    SUM(f.line_total)                                                   AS gross_rev,
    SUM(CASE WHEN f.status <> 'cancelled' THEN f.line_total ELSE 0 END) AS net_rev,
    SUM(f.quantity)                                                     AS units,
    COUNT(DISTINCT f.order_id)                                           AS order_cnt,
    COUNT(*)                                                             AS line_cnt
  FROM b2b_ecommerce.gold.fact_order_items f
  WHERE f.order_date_valid
  GROUP BY f.product_sk
),
per_product AS (
  SELECT
    d.product_id,
    SUM(p.gross_rev) AS gross_revenue,
    SUM(p.net_rev)   AS net_revenue,
    SUM(p.units)     AS units_sold,
    SUM(p.order_cnt) AS order_count,
    SUM(p.line_cnt) AS line_count
  FROM per_sk p
  JOIN b2b_ecommerce.gold.dim_products d ON p.product_sk = d.product_sk
  GROUP BY d.product_id
),
ranked AS (
  SELECT
    p.product_id,
    d.sku            AS sku,
    d.name           AS product_name,
    d.category       AS category,
    d.is_active      AS is_active,
    p.gross_revenue,
    p.net_revenue,
    p.units_sold,
    p.order_count,
    p.line_count,
    ROW_NUMBER() OVER (ORDER BY p.net_revenue DESC, p.product_id) AS net_revenue_rank,
    COUNT(*) OVER ()           AS n_products,
    SUM(p.net_revenue) OVER () AS total_net
  FROM per_product p
  LEFT JOIN b2b_ecommerce.gold.dim_products d
    ON d.product_id = p.product_id AND d.__END_AT IS NULL
)
SELECT
  product_id,
  sku,
  product_name,
  category,
  is_active,
  gross_revenue,
  net_revenue,
  ROUND(100 * net_revenue / total_net, 2) AS net_revenue_share_pct,
  order_count,
  line_count,
  units_sold,
  net_revenue_rank,
  net_revenue_rank <= CEIL(0.2 * n_products) AS is_bestseller,
  net_revenue_rank > FLOOR(0.8 * n_products) AS is_slow_mover
FROM ranked;

In [0]:
%sql
-- Verification: shape, totals, tier flags. Expected: 160 rows = 160 distinct product_id
-- (= full dim catalog, all sold on valid dates), SUM(net) = 28,333,498.48 and
-- SUM(gross) = 30,723,269.53 (exact KPI 1 / KPI 2 reconciliation), bestsellers = 32
-- (= CEIL(0.2 * 160)), slow-movers = 32 (= ranks 129..160), 0 null sku/name/category,
-- ranks 1..160 contiguous.
SELECT
  COUNT(*)                                       AS rows_products,
  COUNT(DISTINCT product_id)                      AS distinct_products,
  SUM(net_revenue)                               AS sum_net,
  SUM(gross_revenue)                             AS sum_gross,
  SUM(order_count)                               AS sum_order_count,
  SUM(CASE WHEN is_bestseller THEN 1 ELSE 0 END)  AS bestseller_count,
  CEIL(0.2 * COUNT(*))                           AS expected_bestsellers,
  SUM(CASE WHEN is_slow_mover THEN 1 ELSE 0 END) AS slow_mover_count,
  COUNT(*) - FLOOR(0.8 * COUNT(*))               AS expected_slow_movers,
  SUM(CASE WHEN sku IS NULL OR product_name IS NULL OR category IS NULL THEN 1 ELSE 0 END) AS null_sku_name_category,
  MIN(net_revenue_rank)                          AS min_rank,
  MAX(net_revenue_rank)                          AS max_rank,
  COUNT(DISTINCT net_revenue_rank)               AS distinct_ranks,
  SUM(CASE WHEN is_bestseller AND is_slow_mover THEN 1 ELSE 0 END) AS flag_overlap
FROM b2b_ecommerce.gold.kpi_3_revenue_by_product_category;

In [0]:
%sql
-- Verification: independently recompute EVERY share % and the rank ordering via
-- self-join (not window). Expected: 160 products checked, share_match = 160,
-- rank_order_match = 160, flag_match = 160.
WITH v AS (
  SELECT product_id, net_revenue, net_revenue_rank, net_revenue_share_pct,
         is_bestseller, is_slow_mover,
         SUM(net_revenue) OVER () AS total_net, COUNT(*) OVER () AS n
  FROM b2b_ecommerce.gold.kpi_3_revenue_by_product_category
),
ord AS (
  SELECT product_id,
         ROW_NUMBER() OVER (ORDER BY net_revenue DESC, product_id) AS expected_rank
  FROM v
)
SELECT
  COUNT(*) AS products_checked,
  SUM(CASE WHEN v.net_revenue_share_pct = ROUND(100 * v.net_revenue / v.total_net, 2) THEN 1 ELSE 0 END) AS share_match,
  SUM(CASE WHEN v.net_revenue_rank = o.expected_rank THEN 1 ELSE 0 END)             AS rank_order_match,
  SUM(CASE WHEN v.is_bestseller  = (v.net_revenue_rank <= CEIL(0.2 * v.n)) THEN 1 ELSE 0 END) AS bestseller_flag_match,
  SUM(CASE WHEN v.is_slow_mover  = (v.net_revenue_rank > FLOOR(0.8 * v.n)) THEN 1 ELSE 0 END) AS slow_mover_flag_match
FROM v
JOIN ord o ON o.product_id = v.product_id;

In [0]:
%sql
-- Verification: category rollup must reproduce the preflight category bases exactly
-- (furniture 11,485,645.18 net / 40.54% ... cleaning 431,865.33 / 1.52%).
SELECT
  category,
  COUNT(*)                                              AS products,
  SUM(net_revenue)                                      AS net_revenue,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS net_share_pct,
  SUM(CASE WHEN is_bestseller THEN 1 ELSE 0 END)        AS bestsellers,
  SUM(CASE WHEN is_slow_mover THEN 1 ELSE 0 END)        AS slow_movers
FROM b2b_ecommerce.gold.kpi_3_revenue_by_product_category
GROUP BY category
ORDER BY net_revenue DESC;

In [0]:
%sql
-- Bestsellers: top-20% products by net revenue (rank 1..32)
SELECT product_id, sku, product_name, category, net_revenue, net_revenue_share_pct,
       units_sold, order_count, net_revenue_rank
FROM b2b_ecommerce.gold.kpi_3_revenue_by_product_category
WHERE is_bestseller
ORDER BY net_revenue_rank;

In [0]:
%sql
-- Slow-movers: bottom-20% products by net revenue (rank 129..160)
SELECT product_id, sku, product_name, category, is_active, net_revenue,
       net_revenue_share_pct, units_sold, order_count, net_revenue_rank
FROM b2b_ecommerce.gold.kpi_3_revenue_by_product_category
WHERE is_slow_mover
ORDER BY net_revenue_rank DESC;